# Notebook 1 — Crawl Berita DetikFinance

Notebook ini **berdiri sendiri** (tanpa import `.py`) dan ditulis **sederhana** (tanpa
class objects) — loop biasa + fungsi pendek, agar mudah dipahami & ditulis ulang.

Class HTML sudah ditemukan di Notebook 0 (pilot): judul `detail__title`,
tanggal `meta publishdate`, isi `detail__body-text`.

## Alur
1. **Indeks per tanggal** → ambil daftar link semua artikel hari itu (judul, url, subkanal).
2. **Filter kandidat** → tandai berita yang judulnya menyebut bank/kebijakan/pasar.
3. **Unduh artikel kandidat** → simpan HTML (`.gz`) + ambil judul/tanggal/isi.

## Output
```
data/raw/html_article/{id[:3]}/{id}.html.gz   <- HTML artikel (1 file = 1 artikel)
data/interim/news_links.parquet (+ .xlsx)      <- daftar semua link + flag kandidat
data/interim/news_clean_raw.parquet            <- judul/tanggal/isi dari kandidat
```
Install: `pip install requests beautifulsoup4 lxml pandas pyarrow openpyxl tqdm`

## 0. Import & pengaturan

In [ ]:
import re, time, gzip, random
from datetime import date, timedelta
from pathlib import Path
import requests
import pandas as pd
from bs4 import BeautifulSoup
from tqdm.auto import tqdm

BASE_URL   = "https://finance.detik.com/indeks"
HEADERS    = {"User-Agent": "Mozilla/5.0 (research crawler; contact: GANTI@email.anda)"}
DELAY      = 1.0            # jeda antar-request (detik) — etika crawling

RAW     = Path("data/raw")
INTERIM = Path("data/interim")
RAW.mkdir(parents=True, exist_ok=True)
INTERIM.mkdir(parents=True, exist_ok=True)
random.seed(42)

## 1. Kata kunci untuk filter kandidat

Daftar kata yang menandakan berita relevan ke bank target. Kalau judul memuat salah satunya
(atau subkanalnya bursa), berita dianggap **kandidat** dan HTML-nya diunduh.

In [ ]:
# kata kunci (huruf kecil) untuk dicek di judul
KEYWORDS = [
    "bbca", "bca", "bank central asia",
    "bbri", "bri", "bank rakyat",
    "bbni", "bni", "bank negara",
    "bmri", "bank mandiri", "mandiri",
    "himbara", "bank bumn", "perbankan",
    "ojk", "bi rate", "suku bunga", "lps", "kur", "npl", "kredit", "dividen",
    "ihsg", "rupiah", "inflasi",
]
BURSA_SUBCHANNELS = ["bursa-dan-valas", "market-research"]

def is_candidate(judul, subkanal):
    """True kalau berita mungkin relevan (judul memuat kata kunci / subkanal bursa)."""
    if subkanal in BURSA_SUBCHANNELS:
        return True
    judul_kecil = (judul or "").lower()
    return any(kata in judul_kecil for kata in KEYWORDS)

## 2. Fungsi-fungsi kecil (ambil halaman, simpan HTML, parse)

Fungsi pendek tanpa class, gaya loop sederhana.

In [ ]:
def ambil_halaman(url):
    """Unduh satu URL, lalu jeda sejenak (etika). Kembalikan HTML string."""
    r = requests.get(url, headers=HEADERS, timeout=30)
    r.raise_for_status()
    time.sleep(DELAY + random.uniform(0.3, 1.0))
    return r.text

def id_dari_url(url):
    """Ambil angka setelah 'd-' sebagai id artikel. Contoh .../d-8688622/... -> '8688622'."""
    return re.findall(r"/d-(\d+)/", url)[0]

def subkanal_dari_url(url):
    """Segmen path pertama, mis. .../moneter/d-123/... -> 'moneter'."""
    hasil = re.findall(r"finance\.detik\.com/([a-z0-9-]+)/d-\d+", url)
    return hasil[0] if hasil else None

def simpan_html(html, path):
    """Simpan HTML sebagai file gzip (hemat ruang)."""
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_bytes(gzip.compress(html.encode("utf-8")))

In [ ]:
def parse_indeks(html):
    """Dari HTML halaman indeks -> daftar artikel (loop sederhana, seperti contoh 'books')."""
    soup = BeautifulSoup(html, "html.parser")
    data = []
    for artikel in soup.find_all("article", class_="list-content__item"):
        link_tag = artikel.find("a", class_="media__link")
        if not link_tag:
            continue
        url = link_tag["href"].split("?")[0]
        if "/d-" not in url:
            continue

        # 1. judul
        judul_tag = artikel.find("h3", class_="media__title") or artikel.find(class_="media__title")
        judul = judul_tag.text.strip() if judul_tag else ""
        # 2. tanggal (teks dari indeks)
        tgl_tag = artikel.find(class_="media__date")
        tgl_teks = tgl_tag.text.strip() if tgl_tag else None

        data.append({
            "article_id": id_dari_url(url),
            "url": url,
            "title": judul,
            "subchannel": subkanal_dari_url(url),
            "date_text": tgl_teks,
        })
    return data

def parse_artikel(html, url):
    """Dari HTML satu artikel -> judul, tanggal, isi (pakai class hasil pilot Notebook 0)."""
    soup = BeautifulSoup(html, "html.parser")

    # 1. judul: <h1 class='detail__title'>
    h1 = soup.find("h1", class_="detail__title")
    judul = h1.text.strip() if h1 else ""

    # 2. tanggal: <meta name='publishdate'>
    meta = soup.find("meta", attrs={"name": "publishdate"})
    tanggal = meta["content"] if meta else None

    # 3. isi: gabung semua <p> di dalam <div class='detail__body-text'>
    body = soup.find("div", class_="detail__body-text")
    isi = ""
    if body:
        paragraf = [p.text.strip() for p in body.find_all("p") if p.text.strip()]
        isi = " ".join(paragraf)

    return {
        "article_id": id_dari_url(url), "url": url, "title": judul,
        "published_at": tanggal, "subchannel": subkanal_dari_url(url),
        "n_words": len(isi.split()), "body_text": isi,
    }

## 3. Loop utama crawl

Untuk tiap tanggal: ambil semua halaman indeks → filter kandidat → unduh artikel kandidat.

In [ ]:
def crawl(tgl_awal, tgl_akhir):
    semua_link = []    # daftar semua artikel (kandidat / bukan)
    artikel_bersih = []  # judul/tanggal/isi dari kandidat saja

    # buat daftar tanggal
    tanggal_list = []
    t = tgl_awal
    while t <= tgl_akhir:
        tanggal_list.append(t)
        t += timedelta(days=1)

    for tgl in tqdm(tanggal_list, desc="crawl per hari"):
        # --- ambil semua halaman indeks hari ini ---
        artikel_hari_ini = []
        for halaman in range(1, 50):
            url_indeks = f"{BASE_URL}?page={halaman}&date={tgl:%m/%d/%Y}"
            html = ambil_halaman(url_indeks)
            if halaman == 1:
                simpan_html(html, RAW / "html_index" / f"{tgl:%Y/%m/%d}" / f"page_{halaman}.html.gz")
            daftar = parse_indeks(html)
            if not daftar:         # halaman kosong -> berhenti
                break
            artikel_hari_ini.extend(daftar)

        # --- filter kandidat & unduh artikel ---
        for art in artikel_hari_ini:
            kandidat = is_candidate(art["title"], art["subchannel"])
            art["index_date"] = tgl.isoformat()
            art["is_candidate"] = kandidat
            art["html_saved"] = False

            if kandidat:
                try:
                    html_art = ambil_halaman(art["url"] + "?single=1")
                    aid = art["article_id"]
                    simpan_html(html_art, RAW / "html_article" / aid[:3] / f"{aid}.html.gz")
                    art["html_saved"] = True
                    isi = parse_artikel(html_art, art["url"])
                    isi["index_date"] = tgl.isoformat()
                    artikel_bersih.append(isi)
                except Exception as e:
                    print("  gagal:", art["url"], "->", e)
            semua_link.append(art)

    return pd.DataFrame(semua_link), pd.DataFrame(artikel_bersih)

## 4. Jalankan — UJI 1 minggu dulu

Ganti dulu email di `HEADERS`. Uji rentang kecil sebelum crawl penuh.

In [ ]:
links, bersih = crawl(date(2024, 1, 2), date(2024, 1, 8))

links.to_parquet(INTERIM / "news_links.parquet", index=False)
links.to_excel(INTERIM / "news_links.xlsx", index=False)
bersih.to_parquet(INTERIM / "news_clean_raw.parquet", index=False)

print("total link    :", len(links))
print("kandidat      :", int(links.is_candidate.sum()))
print("html tersimpan:", int(links.html_saved.sum()))
links.head()

In [ ]:
# ---- RENTANG PENUH (jalankan setelah uji benar; bisa 6-11 jam) ----
# links, bersih = crawl(date(2024, 1, 2), date(2026, 9, 30))
# links.to_parquet(INTERIM / 'news_links.parquet', index=False)
# bersih.to_parquet(INTERIM / 'news_clean_raw.parquet', index=False)

## Catatan
- Versi ini dibuat **sederhana untuk belajar**: tanpa checkpoint & retry otomatis.
  Untuk crawl penuh yang tahan putus, tambahkan checkpoint (simpan progres per tanggal)
  dan retry (ulang bila server error 429/503). Lihat versi lama di `old/` sebagai contoh.
- Isi berita sebagian besar HTML mentah adalah iklan/script; parsing mengambil hanya
  judul/tanggal/isi (class dari Notebook 0).